# Early-exit pipeline, live check

`changed_src/judge_early_exit_pipeline.py` runs the standard queries one at a time and answers early when the T7 judge gives P >= 0.98 to the choice.
Check: run it live (GPU) on official photos from the dump set and compare, photo by photo, with the replay of the same rule on the stored dump
(`Ours_Judge_Check`, section 6). Same stage, same place, same number of queries expected (small differences possible: the dump stores matches at 1/64 px).

In [ ]:
import os, sys, time
os.environ.setdefault("OMP_NUM_THREADS", "6")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT / "tests/Test_Iteration_7/changed_src")); os.chdir(ROOT)
import json, joblib, numpy as np, pandas as pd
from train.checks import replay
from train.checks.candidate_judge import add_context, add_features, decide_frozen, judge_new
T7 = ROOT / "tests/Test_Iteration_7/results"
N_EARLY, N_OTHER = int(os.environ.get("N_EARLY", "8")), int(os.environ.get("N_OTHER", "5"))
table = add_context(add_features(pd.read_csv(T7 / "signals_ours_official150.csv.gz")))
frozen = json.load(open(T7 / "frozen_rules_t7.json")); p = judge_new(joblib.load(T7 / "judge_t7.joblib"), table)
expected = decide_frozen(table, p, frozen["variants"]["pc_early"]["tau"], "pc_early").set_index("photo")
ours = {pid: replay.decide_current(ph) for pid, ph in replay.load_set(T7 / "rich_dump/official150").items()}
early_ids = expected.index[expected.stage == "early"][:N_EARLY].tolist()
other_ids = [i for i in expected.index[expected.stage != "early"] if ours[i]["stage"] in ("escalation", "agreement", "skip")][:N_OTHER] + \
            [i for i in expected.index[expected.stage != "early"] if ours[i]["stage"] == "bar"][:2]
photo_ids = early_ids + other_ids
print(len(early_ids), "photos expected to exit early;", len(other_ids), "others (agreement / escalation / skip / bar)")

In [ ]:
from judge_early_exit_pipeline import JudgeEarlyExitPipeline
pipeline = JudgeEarlyExitPipeline("official", "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=1.0, k_peaks=50, m_fine=10,
                                  matcher="loftr_minima", min_inliers=8, ransac_pixels=None, matcher_confidence=0.22, satellite="new",
                                  map_margin_m=None, loftr_coarse_threshold=0.21, dino_weight=None)
pipeline.setup(500, 7)
for line in pipeline.describe():
    print(line)
test_set = pipeline.test_set

In [ ]:
rows = []
for pid in photo_ids:
    t0 = time.time()
    r = pipeline.run(test_set.items[pid], test_set.priors[pid])
    err = float(np.hypot(r["x"] - test_set.truths[pid][0], r["y"] - test_set.truths[pid][1]) / test_set.sigmas[pid])
    exp = expected.loc[pid]; o = ours[pid]
    rows.append(dict(photo=pid, live_stage=r["stage"], replay_stage=exp.stage if exp.stage == "early" else o["stage"], live_queries=r["n_queries_run"],
                     replay_queries=int(exp.n_queries) if exp.stage == "early" else o["n_queries"], live_err_sigma=round(err, 2),
                     replay_err_sigma=round(float(exp.err_sigma) if exp.stage == "early" else float(o["err_sigma"]), 2), p=round(r["judge_p"], 3),
                     seconds=round(time.time() - t0, 1)))
live = pd.DataFrame(rows)
live["same_stage"] = live.live_stage == live.replay_stage
live["same_place"] = (live.live_err_sigma - live.replay_err_sigma).abs() < 0.2
live

In [ ]:
print("same stage:", int(live.same_stage.sum()), "of", len(live), "| same place:", int(live.same_place.sum()), "of", len(live))
print("mean live queries %.2f vs replay %.2f" % (live.live_queries.mean(), live.replay_queries.mean()))
live.to_csv(T7 / "early_exit_live_check.csv", index=False)